# Floating Point Arithmetic

Let $x = e^h$ with $h = 2^{-n}$, and $y = 1$.
We compare the computed difference $\operatorname{fl}(x) -_{\operatorname{fl}} y$ with $\operatorname{fl}(x - y)$, the exact difference rounded to the nearest double.

In [ ]:
import math

**Caution:** `math.exp` and `math.expm1` might not do the correct rounding, so these values may differ slightly from the lecture notes.

In [ ]:
print(f"{'n':<3} | {'exp(h) - 1':<23} | {'math.expm1(h)':<23} | {'rel. error':<10}")
print("-" * 68)
for n in [*range(26, 33), None, *range(50, 55)]:
    if n is None:
        print(f"{':':<3} | {':':<23} | {':':<23} | {':':<10}")
        continue
    h = 2.0**-n
    naive = math.exp(2**(-n)) - 1
    accurate = math.expm1(h)
    rel = abs(naive - accurate) / abs(accurate)
    print(f"{n:<3d} | {naive:<23.16e} | {accurate:<23.16e} | {rel:<10.3e}")

## Finite differences

### Forward difference quotient

The forward difference approximation of $f'(0) = 1$ for $f(x) = e^x$, with $h = 2^{-n}$, is
$$
f'(c) \approx \frac{f(c + h) - f(c)}{h}.
$$

In [ ]:
def forward_diff_exp(h, c=0.0):
    return (math.exp(c + h) - math.exp(c)) / h

In [ ]:
print(f"{'n':<3} | {'approximation':<23} | {'rel. error':<10}")
print("-" * 42)
for n in [*range(22, 28), None, *range(52, 55)]:
    if n is None:
        print(f"{':':<3} | {':':<23} | {':':<10}")
        continue
    h = 2.0**-n
    approx = forward_diff_exp(h)
    rel = abs(approx - 1)
    print(f"{n:<3d} | {approx:<23.16e} | {rel:<10.3e}")

Any differences from the lecture notes (e.g. for $n = 26$ or $n = 53$) are because `math.exp` is not guaranteed to round correctly.

With $c = 1$, so that $f'(1) = e$:

In [ ]:
print(f"{'n':<3} | {'approximation':<23} | {'rel. error':<10}")
print("-" * 42)
for n in [*range(16, 41, 2), None, *range(50, 54)]:
    if n is None:
        print(f"{':':<3} | {':':<23} | {':':<10}")
        continue
    h = 2.0**-n
    approx = forward_diff_exp(h, c=1.0)
    rel = abs(approx - math.e) / math.e
    print(f"{n:<3d} | {approx:<23.16e} | {rel:<10.3e}")

### Centered difference quotient

The centered difference approximation of $f'(1) = e$ is
$$
f'(c) \approx \frac{f(c + h) - f(c - h)}{2h}.
$$

In [ ]:
def centered_diff_exp(h, c=0.0):
    return (math.exp(c + h) - math.exp(c - h)) / (2 * h)

In [ ]:
print(f"{'n':<3} | {'approximation':<23} | {'rel. error':<10}")
print("-" * 42)
for n in [*range(8, 35, 2), None, *range(50, 54)]:
    if n is None:
        print(f"{':':<3} | {':':<23} | {':':<10}")
        continue
    h = 2.0**-n
    approx = centered_diff_exp(h, c=1.0)
    rel = abs(approx - math.e) / math.e
    print(f"{n:<3d} | {approx:<23.16e} | {rel:<10.3e}")

### Complex step differentiation

The complex step approximation of $f'(1) = e$ is
$$
f'(c) \approx \frac{\operatorname{Im} f(c + ih)}{h},
$$
which has error $\mathcal{O}(h^2)$ and involves no subtraction, so there is no cancellation in the numerator.

In [ ]:
import cmath


def complex_step_exp(h, c=0.0):
    return cmath.exp(c + 1j * h).imag / h

In [ ]:
print(f"{'n':<3} | {'approximation':<23} | {'rel. error':<10}")
print("-" * 42)
for n in [*range(8, 35, 2), None, *range(200, 204)]:
    if n is None:
        print(f"{':':<3} | {':':<23} | {':':<10}")
        continue
    h = 2.0**-n
    approx = complex_step_exp(h, c=1.0)
    rel = abs(approx - math.e) / math.e
    print(f"{n:<3d} | {approx:<23.16e} | {rel:<10.3e}")

## Gaussian Elimination

For $a = 2^{-54}$, we solve
$$
\begin{pmatrix} a & 1 \\ 1 & 1 \end{pmatrix}
\begin{pmatrix} x \\ y \end{pmatrix}
= \begin{pmatrix} 1 \\ 2 \end{pmatrix}
$$
by elimination and back substitution, as in the lecture:
$y = (2 - 1/a) / (1 - 1/a)$ and $x = (1 - y)/a$.
The exact solution is $x = 1/(1 - a)$, $y = (1 - 2a)/(1 - a)$, both very nearly 1.

In [ ]:
a = 2.0**-54
x_exact = 1 / (1 - a)
y_exact = (1 - 2 * a) / (1 - a)

y = (2 - 1 / a) / (1 - 1 / a)
x = (1 - y) / a

print(f"x = {x:<23.16e} rel. error = {abs(x - x_exact) / x_exact:.3e}")
print(f"y = {y:<23.16e} rel. error = {abs(y - y_exact) / y_exact:.3e}")

`np.linalg.solve` uses partial pivoting (swapping the two rows, so that the very small number $a$ is not used as a pivot), which avoids the cancellation:

In [ ]:
import numpy as np

A = np.array([[a, 1.0], [1.0, 1.0]])
b = np.array([1.0, 2.0])
x, y = np.linalg.solve(A, b)

print(f"x = {x:<23.16e} rel. error = {abs(x - x_exact) / x_exact:.3e}")
print(f"y = {y:<23.16e} rel. error = {abs(y - y_exact) / y_exact:.3e}")